# C. Controlled cluster growth in sequence space

You supply one **anchor sequence per cluster** (optionally extra seed sequences for the same cluster). The notebook grows every cluster by repeated profile searches. After each search every hit goes to the cluster it matches clearly better; a hit that two or more clusters match about equally well is **contested**, is kept out of all profiles, and is reported as a boundary sequence. In effect each cluster's radius in sequence space grows until neighbouring clusters touch.

It runs in Google Colab and in a local Jupyter session (Linux, macOS, Windows/Anaconda). Databases come from notebook A.

## What one iteration does
1. **Profile.** The cluster's members are aligned (MAFFT, or a pairwise alignment to the anchor if MAFFT is missing). Only columns where the **anchor has a residue** and that are occupied in more than `MIN_OCCUPANCY` of the members are kept. The anchor therefore defines the region you care about; other domains are ignored. MMseqs2 turns the filtered alignment into a profile.
2. **Search.** All profiles search the database at the E-value of this iteration (`E_SCHEDULE`; later iterations use the last value).
3. **Competition.** A hit goes to its best cluster if that cluster's E-value beats every other cluster's by at least `MARGIN` orders of magnitude. Otherwise it is contested.
4. **Report.** Cluster sizes, new / lost / contested counts, FASTA files per cluster, and a plot of within-cluster versus cross-cluster distances with the newest members highlighted.

There is no stopping rule. `ITERATIONS` is the number to have completed; if more are asked for than are saved in the checkpoints, the missing ones are run and the rest are loaded.

## Steps
| Step | What happens | Typical cost |
|---|---|---|
| 1 Environment | Finds or installs MMseqs2, MAFFT, Biopython | A minute or two (not measured) |
| 2 Database | Loads a notebook A database | Seconds to minutes (copy) |
| 3 Seeds | Reads your FASTA | Instant |
| 4 Verify input | Integrity checks, all pairwise alignments of your sequences, identity map | Seconds |
| 5 Setup / pool | Candidate pool: one loose search of the whole database with all seeds (or nothing, in full-database mode) | Like one search in notebook B (hours on 5.7 M sequences; not measured) |
| 6 Iterations | Profile, search, competition, plot, per iteration | Seconds to minutes in pool mode (not measured); a full-database search each time in full mode |
| 7 Final check | Pool mode only: searches the full database once with the final profiles | One full-database search |
| 8 Output | FASTA per cluster, tables, zip | Seconds |

## Things to keep in mind
- **Profile E-values from different clusters are not directly comparable** in a strict statistical sense (different profiles, different lengths). The margin rule treats a difference of `MARGIN` orders of magnitude as a clear preference. Check the distance plots rather than trusting the rule blindly.
- **In pool mode E-values are corrected** for the smaller target set (multiplied by the ratio of full database size to pool size in residues), so they mean what they would mean in the full database.
- **Drift** is the main failure mode of iterative searches: a wrong sequence enters a profile and pulls in more wrong sequences. Contested sequences never enter profiles, and `PROFILE_MAX_EVALUE` keeps weak members out of the profile. Watch the "own versus nearest other cluster" plot: points below the diagonal are closer to another cluster than to their own.
- **UniRef50** entries are cluster representatives, so counts are not species counts.
- **Windows.** The code uses no shell features and no Linux-only paths, but MMseqs2 supports native Windows only as a beta and MAFFT needs its own Windows install. The reliable route is WSL2 with a conda environment (the setup steps are at the top of notebook A). Nothing here has been tested on Windows; the logic has been tested on Linux only.

In [ ]:
#@title Parameters { display-mode: "form" }
#@markdown ### Seeds (the clusters to grow)
#@markdown `SEEDS_FASTA`: path to a multi-FASTA. Header format: `>label name` or `>label|name`. Sequences with the same label form one cluster; **the first sequence of a label is its anchor**. In Colab you can tick `UPLOAD_SEEDS` instead of giving a path. (Local Jupyter users can also paste the FASTA into `SEEDS_INLINE` in the Advanced cell.)
SEEDS_FASTA = "" #@param {type:"string"}
UPLOAD_SEEDS = False #@param {type:"boolean"}
#@markdown ### Database (made in notebook A)
DATABASE = "" #@param {type:"string"}
LOCAL_COPY = True #@param {type:"boolean"}
#@markdown Folders. Empty = a sensible default next to the notebook (or under /content in Colab).
WORKDIR = "" #@param {type:"string"}
LOCAL_DIR = "" #@param {type:"string"}
OUT_DIR = "" #@param {type:"string"}
#@markdown ### Tools
#@markdown Leave empty to use what is on the PATH or let the notebook install it. On Windows give the full path to `mmseqs.bat`/`mmseqs.exe` and `mafft.bat` if they are not on the PATH.
MMSEQS_EXE = "" #@param {type:"string"}
MAFFT_EXE = "" #@param {type:"string"}
AUTO_INSTALL = True #@param {type:"boolean"}
#@markdown ### Search scope
#@markdown `candidate pool first`: one loose search of the full database with all seeds, then all iterations inside that pool (fast), then one final check against the full database. `full database every iteration`: every iteration searches the whole database (slow, most complete; `BUILD_INDEX` makes repeated searches cheaper).
SEARCH_SCOPE = "candidate pool first" #@param ["candidate pool first", "full database every iteration"]
POOL_EVALUE = 10.0 #@param {type:"number"}
POOL_MAX_SEQS_PER_SEED = 20000 #@param {type:"integer"}
POOL_SENSITIVITY = 7.5 #@param {type:"number"}
POOL_ITERATIONS = 2 #@param {type:"integer"}
BUILD_INDEX = False #@param {type:"boolean"}
SPLIT_MEMORY_LIMIT = "8G" #@param {type:"string"}
#@markdown ### Growth
ITERATIONS = 6 #@param {type:"integer"}
E_SCHEDULE = "1e-10,1e-8,1e-6,1e-4,1e-3,1e-2" #@param {type:"string"}
#@markdown `MARGIN`: orders of magnitude by which the best cluster's E-value must beat all others to win a sequence.
MARGIN = 3.0 #@param {type:"number"}
#@markdown `PROFILE_MAX_EVALUE`: only members at least this good enter the profile (0 = no limit, riskier). `MIN_OCCUPANCY`: alignment columns need residues in more than this fraction of members. `MIN_PROFILE_COVERAGE`: a hit must cover at least this fraction of the profile.
PROFILE_MAX_EVALUE = 1e-4 #@param {type:"number"}
MIN_OCCUPANCY = 0.10 #@param {type:"number"}
MIN_PROFILE_COVERAGE = 0.5 #@param {type:"number"}
PROFILE_SENSITIVITY = 7.5 #@param {type:"number"}
MAX_HITS_PER_PROFILE = 50000 #@param {type:"integer"}
MAX_PROFILE_SEQS = 1000 #@param {type:"integer"}
#@markdown ### Alignment for the profiles
MSA_METHOD = "auto" #@param ["auto", "mafft", "pairwise to anchor (no MAFFT needed)"]
MAFFT_MODE = "fast" #@param ["fast", "auto", "accurate"]
#@markdown ### Distances and plots
DIST_METRIC = "identity" #@param ["identity", "normalised score"]
DIST_MATRIX = "blosum62.out" #@param ["blosum62.out", "blosum45.out", "blosum80.out", "PAM30.out", "PAM70.out", "VTML80.out"]
DIST_SAMPLE = 120 #@param {type:"integer"}
SHOW_ALL_ITERATION_PLOTS = False #@param {type:"boolean"}
#@markdown ### Input verification
#@markdown The notebook stops after verifying your input until `CONFIRM_INPUT` is ticked. `SHOW_ALL_PAIRS` shows every pairwise alignment, otherwise only anchor-versus-anchor and seed-versus-anchor.
CONFIRM_INPUT = False #@param {type:"boolean"}
SHOW_ALL_PAIRS = False #@param {type:"boolean"}
#@markdown ### Output
FASTA_REGION_ONLY = False #@param {type:"boolean"}
FINAL_CHECK = True #@param {type:"boolean"}
#@markdown ### Checkpoints
CKPT_DIR = "" #@param {type:"string"}
DOWNLOAD_CKPT = True #@param {type:"boolean"}
UPLOAD_CKPT = False #@param {type:"boolean"}
OVERRIDE_CHECKPOINT = False #@param {type:"boolean"}
FORCE_RERUN = False #@param {type:"boolean"}
KEEP_HISTORY = False #@param {type:"boolean"}

In [ ]:
# Advanced (optional)
# Paste the seed FASTA here instead of using a file (used only when SEEDS_FASTA is empty and nothing was uploaded).
SEEDS_INLINE = """"""

## 1. Environment
**What happens.** Finds MMseqs2 and MAFFT, or installs them where that is possible, and imports Biopython, pandas, numpy and matplotlib. Commands are run without a shell, so the same code works on Linux, macOS and Windows.

- **MMseqs2:** `MMSEQS_EXE`, else the PATH, else a copy downloaded from the official GitHub releases (Linux and macOS; on Windows an attempt is made, but see the Windows note above and use WSL2 if it fails).
- **MAFFT:** `MAFFT_EXE`, else the PATH, else `apt-get` on a Linux machine where that is allowed (Colab). On Windows download MAFFT from the MAFFT website (the Windows zip) or use WSL; on Anaconda for Linux/macOS: `conda install -c bioconda mafft`. Without MAFFT the notebook can still run with `MSA_METHOD = pairwise to anchor`, which aligns each member to the anchor on its own. That is lower quality for remote homologs but needs no external program.

**Expect.** A minute or two if something has to be installed (not measured). Output: tool versions, threads, free disk.

In [ ]:
#@title 1. Environment { display-mode: "form" }
import os, sys, re, json, math, shutil, hashlib, subprocess, time, platform, gzip, datetime, struct, tarfile, zipfile, itertools, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SYSTEM = platform.system()                       # Linux, Darwin, Windows
THREADS = os.cpu_count() or 2
_base = Path("/content") if IN_COLAB else Path.cwd()
WORK = Path(WORKDIR) if WORKDIR else _base / "work_C"
WORK.mkdir(parents=True, exist_ok=True)
TMP = WORK / "tmp"; TMP.mkdir(exist_ok=True)
LOCAL_DIR = LOCAL_DIR or str(_base / "mmseqs_local")
OUT = Path(OUT_DIR) if OUT_DIR else WORK / "results"
OUT.mkdir(parents=True, exist_ok=True)
BIN = WORK / "bin"
PROGRESS_RE = re.compile(r"^\s*\[[=> ]*\]")

def run(args, stdout_path=None, quiet=False, tail=25, every=20):
    """Run a program (list of arguments, no shell). MMseqs2 progress-bar lines are shown at most once per `every` seconds.
    With stdout_path the program's stdout goes to that file (MAFFT writes its alignment to stdout)."""
    args = [str(a) for a in args]
    if not quiet:
        print("$", " ".join(f'"{a}"' if " " in a else a for a in args)[:300])
    t0, last, lines = time.time(), 0.0, []
    fh = open(stdout_path, "w", newline="\n") if stdout_path else None
    try:
        p = subprocess.Popen(args, stdout=fh if fh else subprocess.PIPE, stderr=subprocess.PIPE if fh else subprocess.STDOUT,
                             text=True, bufsize=1)
    except OSError as e:
        raise RuntimeError(f"Could not start {args[0]!r}: {e}")
    for line in (p.stderr if fh else p.stdout):
        lines.append(line)
        if quiet:
            continue
        if PROGRESS_RE.match(line):
            if time.time() - last >= every:
                print(f"  [{time.time() - t0:6.0f}s] {line.strip()[:110]}"); last = time.time()
        else:
            print(line, end="")
    rc = p.wait()
    if fh: fh.close()
    if rc:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(args)[:300]}\n" + "".join(lines[-tail:]))
    if not quiet:
        print(f"  (finished in {time.time() - t0:.0f}s)")
    return "".join(lines)

def _find(names, extra_dirs=()):
    for n in names:
        w = shutil.which(n)
        if w: return w
    for d in extra_dirs:
        if Path(d).exists():
            for n in names:
                for p in Path(d).rglob(n):
                    if p.is_file(): return str(p)
    return None

def find_mmseqs():
    if MMSEQS_EXE:
        if not Path(MMSEQS_EXE).exists(): raise FileNotFoundError(f"MMSEQS_EXE {MMSEQS_EXE!r} does not exist.")
        return MMSEQS_EXE
    return _find(["mmseqs", "mmseqs.bat", "mmseqs.exe"], [BIN])

def install_mmseqs():
    BIN.mkdir(parents=True, exist_ok=True)
    base = "https://github.com/soedinglab/MMseqs2/releases/latest/download/"
    if SYSTEM == "Linux":
        if platform.machine().lower() in ("aarch64", "arm64"):
            asset = "mmseqs-linux-arm64.tar.gz"
        else:
            try: flags = open("/proc/cpuinfo").read()
            except OSError: flags = ""
            asset = "mmseqs-linux-" + ("avx2" if " avx2" in flags else ("sse41" if " sse4_1" in flags else "sse2")) + ".tar.gz"
    elif SYSTEM == "Darwin":
        asset = "mmseqs-osx-universal.tar.gz"
    elif SYSTEM == "Windows":
        asset = "mmseqs-win64.zip"
    else:
        raise RuntimeError(f"No MMseqs2 download known for {SYSTEM}; install it and set MMSEQS_EXE.")
    dest = BIN / asset
    print("Downloading", base + asset)
    try:
        urllib.request.urlretrieve(base + asset, dest)
    except Exception as e:
        raise RuntimeError(f"Could not download {asset} ({e}). " + ("MMseqs2 supports Windows only as a beta and the current release may not offer a Windows file. "
                           "Recommended: run this notebook inside WSL2 (see notebook A for the steps), or install MMseqs2 yourself and set MMSEQS_EXE." if SYSTEM == "Windows"
                           else "Install MMseqs2 yourself and set MMSEQS_EXE."))
    if asset.endswith(".zip"):
        with zipfile.ZipFile(dest) as z: z.extractall(BIN)
    else:
        with tarfile.open(dest) as t: t.extractall(BIN)
    dest.unlink()

MMSEQS = find_mmseqs()
if not MMSEQS and AUTO_INSTALL:
    install_mmseqs(); MMSEQS = find_mmseqs()
if not MMSEQS:
    raise RuntimeError("MMseqs2 not found. Install it (https://github.com/soedinglab/MMseqs2/releases) and set MMSEQS_EXE, or tick AUTO_INSTALL.")

def mm(*args, **kw):
    return run([MMSEQS, *args], **kw)

MAFFT = MAFFT_EXE or _find(["mafft", "mafft.bat"])
if not MAFFT and AUTO_INSTALL and SYSTEM == "Linux" and shutil.which("apt-get") and hasattr(os, "geteuid") and os.geteuid() == 0:
    try:
        run(["apt-get", "update", "-qq"], quiet=True); run(["apt-get", "install", "-y", "-qq", "mafft"], quiet=True)
        MAFFT = _find(["mafft"])
    except RuntimeError as e:
        print("MAFFT installation failed:", str(e)[:200])
USE_MAFFT = (MSA_METHOD == "mafft") or (MSA_METHOD == "auto" and bool(MAFFT))
if USE_MAFFT and not MAFFT:
    raise RuntimeError("MSA_METHOD = mafft but MAFFT was not found. Install it, set MAFFT_EXE, or use 'pairwise to anchor'.")
if MSA_METHOD == "auto" and not MAFFT:
    print("NOTE: MAFFT not found; profiles will be built from pairwise alignments to the anchor (see the notes above step 1).")

try:
    import Bio
except ImportError:
    run([sys.executable, "-m", "pip", "install", "-q", "biopython"], quiet=True)
    import Bio
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
from Bio import Align
from Bio.Align import substitution_matrices

print(f"{SYSTEM} | python {platform.python_version()} | biopython {Bio.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print("mmseqs:", MMSEQS, "|", mm("version", quiet=True).strip().splitlines()[-1])
print("alignment method:", "MAFFT (" + str(MAFFT) + ", mode " + MAFFT_MODE + ")" if USE_MAFFT else "pairwise to anchor (Biopython)")
print("threads:", THREADS, "| free disk (GB) in work dir:", round(shutil.disk_usage(WORK).free / 1e9, 1))
print("work:", WORK, "| results:", OUT)

## 1b. Checkpoint helpers
**What happens.** Defines how steps save and reload results. Each step writes `CKPT_DIR/<step>/meta.json` (its parameters, their hash, a timestamp) and data files. A step's parameters include the key of the checkpoint it was built from, so changing a setting invalidates everything after it. In Colab, `UPLOAD_CKPT` restores zips downloaded earlier; locally just keep the folder.

**Expect.** Instant. Output lists the steps already saved.

**Parameters that matter.**
- `OVERRIDE_CHECKPOINT`: load a checkpoint although its parameters differ (results then belong to the old parameters).
- `FORCE_RERUN`: ignore checkpoints and recompute.
- `KEEP_HISTORY`: move old checkpoints to `CKPT_DIR/history` instead of deleting.
- **Continuing:** raise `ITERATIONS` and run again; finished iterations are loaded, new ones are computed.

In [ ]:
#@title 1b. Checkpoint helpers { display-mode: "form" }
NB_VERSION = "C1"
CK = Path(CKPT_DIR) if CKPT_DIR else WORK / "checkpoints"
CK.mkdir(parents=True, exist_ok=True)
KEYS = {}

def keyof(*objs):
    return hashlib.sha1(json.dumps(objs, sort_keys=True, default=str).encode()).hexdigest()[:12]

def _short(v, n=70):
    s = str(v)
    return s if len(s) <= n else s[:n] + f"... ({len(s)} characters)"

def ck_load(step, params, expensive=False):
    mf = CK / step / "meta.json"
    if FORCE_RERUN:
        print(f"[{step}] FORCE_RERUN is ticked: recomputing and overwriting."); return None
    if not mf.exists():
        return None
    meta = json.loads(mf.read_text())
    if meta["key"] == keyof(params):
        print(f"[{step}] using checkpoint from {meta['created']} (parameters match)")
        KEYS[step] = meta["key"]; return meta
    old = meta["params"]
    diff = [f"    {k}: checkpoint={_short(old.get(k))} | now={_short(params.get(k))}"
            for k in sorted(set(old) | set(params)) if old.get(k) != params.get(k)]
    head = f"[{step}] a checkpoint from {meta['created']} exists but was made with different parameters:\n" + "\n".join(diff)
    if OVERRIDE_CHECKPOINT:
        print(head + "\n    OVERRIDE_CHECKPOINT is ticked: loading the old checkpoint anyway. Results below correspond to the OLD parameters.")
        KEYS[step] = meta["key"]; return meta
    if expensive:
        raise RuntimeError(head + "\n    This step is slow, so it stops instead of recomputing. Tick OVERRIDE_CHECKPOINT to load the old "
                           "checkpoint anyway, or FORCE_RERUN to recompute and overwrite it.")
    print(head + "\n    Recomputing; the old checkpoint will be " + ("moved to history." if KEEP_HISTORY else "overwritten."))
    return None

def ck_table(step, name):
    return pd.read_csv(CK / step / f"{name}.tsv.gz", sep="\t", dtype=str, keep_default_na=False)

def ck_save(step, params, tables=None, files=None, extra=None):
    d = CK / step
    if d.exists():
        if KEEP_HISTORY:
            (CK / "history").mkdir(exist_ok=True)
            shutil.move(str(d), str(CK / "history" / f"{step}_{datetime.datetime.now():%Y%m%d_%H%M%S}"))
        else:
            shutil.rmtree(d)
    d.mkdir(parents=True)
    for n, df in (tables or {}).items():
        df.to_csv(d / f"{n}.tsv.gz", sep="\t", index=False)
    for n, p in (files or {}).items():
        shutil.copy(p, d / n)
    meta = {"step": step, "notebook_version": NB_VERSION, "created": datetime.datetime.now().isoformat(timespec="seconds"),
            "key": keyof(params), "params": params, "extra": extra or {}}
    (d / "meta.json").write_text(json.dumps(meta, indent=1, default=str))
    KEYS[step] = meta["key"]
    man = [json.loads(m.read_text()) for m in sorted(CK.glob("*/meta.json"))]
    (CK / "manifest.json").write_text(json.dumps([{k: m[k] for k in ("step", "created", "key", "notebook_version")} for m in man], indent=1))
    print(f"[{step}] checkpoint saved: {d}")
    if DOWNLOAD_CKPT and IN_COLAB:
        z = shutil.make_archive(str(WORK / f"ckpt_{step}"), "zip", str(CK), step)
        print(f"[{step}] zip: {z} ({os.path.getsize(z) / 1e6:.1f} MB)")
        from google.colab import files as colab_files
        colab_files.download(z)

if UPLOAD_CKPT:
    if not IN_COLAB:
        print("UPLOAD_CKPT only works in Colab; copy the unzipped step folders into", CK, "instead.")
    else:
        from google.colab import files as colab_files
        for fname in colab_files.upload():
            if fname.endswith(".zip"):
                with zipfile.ZipFile(fname) as z: z.extractall(CK)
                print("restored", fname)
print("Checkpoint folder:", CK, "| existing steps:", sorted(p.parent.name for p in CK.glob("*/meta.json")) or "none")

## 2. Load the database
**What happens.** Reads `db_info.json` in the database folder (or unpacks a `.tar` package into `LOCAL_DIR`), checks that all files are present, that none is a link and that the sequence count matches, copies a database on Google Drive to local disk, and prints what it is. Databases made on Linux in Colab can be used on Windows if the MMseqs2 versions are compatible (same major release).

**Expect.** Seconds for a local folder; copying or unpacking takes as long as the data size at your transfer speed (not measured).

**Parameters that matter.** `DATABASE` decides which taxa can be found. `LOCAL_COPY` copies from Drive first.

In [ ]:
#@title 2. Load the database { display-mode: "form" }
def n_seqs(db):
    return sum(1 for _ in open(str(db) + ".index"))

def db_residues(db):
    tot = 0
    with open(str(db) + ".index") as f:
        for line in f:
            tot += int(line.split()[2]) - 2          # length includes newline and terminating zero
    return tot

def load_database(spec, label="database"):
    if not spec:
        raise ValueError(f"{label} is empty. Set it to a database folder (or .tar) made by notebook A.")
    p = Path(spec)
    if p.is_file() and p.suffix == ".tar":
        with tarfile.open(p) as tf:
            top = tf.getnames()[0].split("/")[0]
            dest = Path(LOCAL_DIR) / top
            if (dest / "db_info.json").exists():
                print(f"{p.name}: already unpacked in {dest}")
            else:
                Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)
                tf.extractall(LOCAL_DIR)
                print(f"{p.name}: unpacked to {dest}")
        p = dest
    if not (p / "db_info.json").exists():
        raise FileNotFoundError(f"No db_info.json in {p}. Use a folder made by notebook A.")
    if str(p.resolve()).startswith("/content/drive") and LOCAL_COPY:
        dest = Path(LOCAL_DIR) / p.name
        if (dest / "db_info.json").exists():
            print(f"Using the local copy {dest}")
        else:
            Path(LOCAL_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Copying {p} to {dest} (searching straight from Drive is slow)")
            shutil.copytree(p, dest)
        p = dest
    info = json.loads((p / "db_info.json").read_text())
    db = p / info.get("db_prefix", "db")
    for ext in (".dbtype", ".index", ".lookup", "_h", "_h.index"):
        if not Path(str(db) + ext).exists():
            raise FileNotFoundError(f"{p} is missing db{ext}.")
    broken = [f.name for f in p.iterdir() if f.is_symlink()]
    if broken:
        raise RuntimeError(f"{p} contains links ({broken[:3]}...); it is not a standalone database. Rebuild it with notebook A.")
    n = n_seqs(db)
    if n != info["sequences"]:
        raise RuntimeError(f"{p}: the index lists {n:,} sequences but db_info.json says {info['sequences']:,}; the copy is incomplete.")
    print(f"{label}: {info['name']} ({info['kind']}), {n:,} sequences, {info['distinct_taxids']:,} distinct taxids, made {info['created']} with {info['notebook']}")
    print(f"  source: {info['source']} | taxids: {info['taxids'] or 'n/a'} | self-test: {info['checks']['self_test']}")
    return db, info

DB, INFO = load_database(DATABASE, "DATABASE")
DB_RESIDUES = db_residues(DB)
print(f"  {DB_RESIDUES:,} residues in total")

## 3. Read the seeds
**What happens.** Reads the multi-FASTA (from `SEEDS_FASTA`, an upload in Colab, or `SEEDS_INLINE`). The header gives the cluster label (`>label name` or `>label|name`). The first sequence of each label is that cluster's **anchor**, the others are extra seeds. Clusters are numbered 1, 2, 3 ... in order of first appearance; these numbers appear in the names of contested sequences (`contested_a2_...` means "also matched by cluster 2 about equally well").

**Expect.** Instant. Output: a table with one row per sequence.

**Parameters that matter.** The order of the labels in the file defines the cluster numbers. Put the sequence that defines the region you care about first in each label.

In [ ]:
#@title 3. Read the seeds { display-mode: "form" }
def read_text_seeds():
    if UPLOAD_SEEDS:
        if not IN_COLAB:
            raise RuntimeError("UPLOAD_SEEDS only works in Colab; give a path in SEEDS_FASTA.")
        from google.colab import files as colab_files
        up = colab_files.upload()
        return list(up.values())[0].decode()
    if SEEDS_FASTA:
        p = Path(SEEDS_FASTA)
        if not p.exists(): raise FileNotFoundError(f"SEEDS_FASTA {p} does not exist.")
        return p.read_text()
    if SEEDS_INLINE.strip():
        return SEEDS_INLINE
    raise ValueError("No seeds. Set SEEDS_FASTA, tick UPLOAD_SEEDS, or paste a FASTA into SEEDS_INLINE.")

def clean_name(s):
    return re.sub(r"[^\w.\-]+", "_", s).strip("_")

def parse_seeds(text):
    recs, hdr, buf = [], None, []
    for line in text.splitlines():
        if line.startswith(">"):
            if hdr is not None: recs.append((hdr, "".join(buf)))
            hdr, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if hdr is not None: recs.append((hdr, "".join(buf)))
    rows, order, count = [], [], {}
    for hdr, seq in recs:
        first, _, rest = hdr.partition(" ")
        if "|" in first:
            label, _, nm = first.partition("|"); nm = (nm + " " + rest).strip()
        else:
            label, nm = first, rest.strip()
        label = clean_name(label)
        if not label: raise ValueError(f"Header '>{hdr}' has no cluster label.")
        if label not in order: order.append(label)
        count[label] = count.get(label, 0) + 1
        role = "anchor" if count[label] == 1 else "seed"
        name = clean_name(nm) or (f"{label}_anchor" if role == "anchor" else f"{label}_seed{count[label] - 1}")
        seq = re.sub(r"\s+", "", seq).upper().rstrip("*")
        rows.append({"cluster": label, "cnum": order.index(label) + 1, "role": role, "name": name, "seq": seq})
    return pd.DataFrame(rows)

SEEDS = parse_seeds(read_text_seeds())
CLUSTERS = list(dict.fromkeys(SEEDS.cluster))
CNUM = {c: i + 1 for i, c in enumerate(CLUSTERS)}
ANCHOR = {c: SEEDS[(SEEDS.cluster == c) & (SEEDS.role == "anchor")].seq.iloc[0] for c in CLUSTERS}
SEEDS_KEY = keyof(list(zip(SEEDS.cluster, SEEDS.name, SEEDS.seq)))
view = SEEDS.assign(length=SEEDS.seq.str.len())[["cnum", "cluster", "role", "name", "length"]]
display(view)

## 4. Verify input
**What happens.** Checks that the input is what you think it is, before anything expensive runs:
- **Integrity:** only amino-acid letters, plausible lengths, no empty or duplicate names, no identical sequence in two clusters, at least two clusters (competition needs at least two; with one cluster the notebook still runs but nothing can be contested).
- **Warnings:** anchors of different clusters that are very similar (they will be hard to separate), a seed that does not resemble its own anchor (wrong label?), low-complexity or very short sequences.
- **Pairwise alignments:** a global alignment (BLOSUM62, gap open 10 / extend 0.5) of every anchor pair and of every seed against its anchor, with identity and score, printed in blocks (`SHOW_ALL_PAIRS` prints every pair). Between lines: `|` identical, `:` similar, space different.
- An **identity map** of all your sequences.

The notebook then stops until you tick `CONFIRM_INPUT`. That tick is your confirmation that these are the right sequences, in the right clusters, in the right order.

**Expect.** Seconds. A fatal problem stops the notebook with a list; warnings do not.

In [ ]:
#@title 4. Verify input { display-mode: "form" }
problems, warnings_ = [], []
AA = set("ACDEFGHIKLMNPQRSTVWYXBZUO")
for _, r in SEEDS.iterrows():
    bad = sorted(set(r.seq) - AA)
    if not r.seq: problems.append(f"{r['name']}: empty sequence")
    if bad: problems.append(f"{r['name']}: characters that are not amino-acid letters: {bad}")
    if 0 < len(r.seq) < 30: warnings_.append(f"{r['name']}: only {len(r.seq)} residues")
    if len(r.seq) > 3000: warnings_.append(f"{r['name']}: {len(r.seq)} residues (very long; is this the right sequence?)")
    if r.seq and r.seq.count("X") / len(r.seq) > 0.1: warnings_.append(f"{r['name']}: more than 10% X")
    if r.seq and max(r.seq.count(a) for a in set(r.seq)) / len(r.seq) > 0.4: warnings_.append(f"{r['name']}: one residue type makes up over 40% (low complexity?)")
if problems:
    raise RuntimeError("Input problems (fix these first; the alignments are not shown until the sequences are valid):\n  - " + "\n  - ".join(problems))
dups = SEEDS[SEEDS.duplicated("name", keep=False)]
if len(dups): problems.append(f"duplicate names: {sorted(set(dups.name))}")
for s, g in SEEDS.groupby("seq"):
    if len(g) > 1 and len(set(g.cluster)) > 1:
        problems.append(f"identical sequence in several clusters: {list(g.name)}")
    elif len(g) > 1:
        warnings_.append(f"identical sequences within cluster {g.cluster.iloc[0]}: {list(g.name)}")
if len(CLUSTERS) < 2:
    warnings_.append("only one cluster: nothing can be contested; the run just grows that cluster")
if len(CLUSTERS) > 9:
    warnings_.append("more than 9 clusters: contested names use '_' between cluster numbers (e.g. a2_11)")
try:
    ES = [float(x) for x in E_SCHEDULE.replace(";", ",").split(",") if x.strip()]
    if not ES or any(e <= 0 for e in ES): raise ValueError
except ValueError:
    ES = []; problems.append(f"E_SCHEDULE '{E_SCHEDULE}' is not a comma-separated list of positive numbers")
if ITERATIONS < 1: problems.append("ITERATIONS must be at least 1")
if MARGIN <= 0: problems.append("MARGIN must be positive")
if not 0 <= MIN_OCCUPANCY < 1: problems.append("MIN_OCCUPANCY must be between 0 and 1")
if PROFILE_MAX_EVALUE <= 0:
    warnings_.append("PROFILE_MAX_EVALUE = 0: every assigned member enters the profile, including weak ones. Drift (a wrong sequence pulling in more wrong ones) is then more likely.")
elif ES and PROFILE_MAX_EVALUE > max(ES):
    warnings_.append("PROFILE_MAX_EVALUE is looser than the whole E_SCHEDULE, so it has no effect: all members enter the profile.")

PA = Align.PairwiseAligner()
PA.substitution_matrix = substitution_matrices.load("BLOSUM62")
PA.mode = "global"; PA.open_gap_score = -10; PA.extend_gap_score = -0.5

def pair_aln(a, b):
    """Global alignment; returns (identity over the shorter sequence, identity over aligned columns, score, rows a/mid/b)."""
    fix = lambda x: re.sub(r"[^ACDEFGHIKLMNPQRSTVWYX]", "X", x.replace("U", "C").replace("O", "K"))
    aln = PA.align(fix(a), fix(b))[0]
    ra, rb, mid, ident, cols = [], [], [], 0, 0
    pa = pb = 0
    M = substitution_matrices.load("BLOSUM62")
    for (a0, a1), (b0, b1) in zip(*aln.aligned):
        if a0 > pa or b0 > pb:                           # gap before this block
            ga, gb = a[pa:a0], b[pb:b0]
            ra.append(ga + "-" * len(gb)); rb.append("-" * len(ga) + gb); mid.append(" " * (len(ga) + len(gb)))
        for x, y in zip(a[a0:a1], b[b0:b1]):
            ra.append(x); rb.append(y); cols += 1
            if x == y: ident += 1; mid.append("|")
            else: mid.append(":" if (x in M.alphabet and y in M.alphabet and M[x, y] > 0) else " ")
        pa, pb = a1, b1
    if pa < len(a) or pb < len(b):
        ga, gb = a[pa:], b[pb:]
        ra.append(ga + "-" * len(gb)); rb.append("-" * len(ga) + gb); mid.append(" " * (len(ga) + len(gb)))
    ra, rb, mid = "".join(ra), "".join(rb), "".join(mid)
    return ident / max(1, min(len(a), len(b))), ident / max(1, cols), aln.score, (ra, mid, rb)

def show_alignment(na, nb_, a, b, width=90):
    idn, idc, sc, (ra, mid, rb) = pair_aln(a, b)
    print(f"\n{na} ({len(a)}) vs {nb_} ({len(b)}): identity {idn:.0%} of the shorter sequence, {idc:.0%} of aligned columns, score {sc:.0f}")
    for i in range(0, len(ra), width):
        print(f"  {ra[i:i+width]}\n  {mid[i:i+width]}\n  {rb[i:i+width]}\n")
    return idn

names, seqs = list(SEEDS.name), list(SEEDS.seq)
N = len(names)
IDM = np.eye(N)
for i in range(N):
    for j in range(i + 1, N):
        IDM[i, j] = IDM[j, i] = pair_aln(seqs[i], seqs[j])[0] if seqs[i] and seqs[j] else 0

anchors_idx = [i for i in range(N) if SEEDS.role.iloc[i] == "anchor"]
pairs = [(i, j) for i, j in itertools.combinations(anchors_idx, 2)]
pairs += [(i, j) for i in anchors_idx for j in range(N) if SEEDS.role.iloc[j] == "seed" and SEEDS.cluster.iloc[j] == SEEDS.cluster.iloc[i]]
if SHOW_ALL_PAIRS:
    pairs = list(itertools.combinations(range(N), 2))
print(f"=== Pairwise alignments ({len(pairs)} pairs) ===")
for i, j in pairs:
    show_alignment(names[i], names[j], seqs[i], seqs[j])
for i, j in itertools.combinations(anchors_idx, 2):
    if IDM[i, j] > 0.9: warnings_.append(f"anchors {names[i]} and {names[j]} are {IDM[i, j]:.0%} identical: their clusters will be almost impossible to separate")
for j in range(N):
    if SEEDS.role.iloc[j] == "seed":
        i = [k for k in anchors_idx if SEEDS.cluster.iloc[k] == SEEDS.cluster.iloc[j]][0]
        if IDM[i, j] < 0.2: warnings_.append(f"seed {names[j]} is only {IDM[i, j]:.0%} identical to the anchor of its cluster ({names[i]}); wrong label?")

fig, ax = plt.subplots(figsize=(max(4, 0.45 * N + 2), max(3.5, 0.45 * N + 1.5)))
im = ax.imshow(IDM, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(N)); ax.set_xticklabels([f"{n}" for n in names], rotation=90, fontsize=7)
ax.set_yticks(range(N)); ax.set_yticklabels([f"{c}:{n}" for c, n in zip(SEEDS.cluster, names)], fontsize=7)
for k in range(N):
    for l in range(N):
        if N <= 14: ax.text(l, k, f"{IDM[k, l]:.0%}", ha="center", va="center", fontsize=6, color="white" if IDM[k, l] < 0.6 else "black")
plt.colorbar(im, label="identity (of shorter sequence)"); ax.set_title("Identity between your input sequences")
plt.tight_layout(); plt.show()

print("\n=== Verdict ===")
for w in warnings_: print("WARNING:", w)
if problems:
    raise RuntimeError("Input problems:\n  - " + "\n  - ".join(problems))
print(f"{len(CLUSTERS)} clusters, {N} sequences; no fatal problems; {len(warnings_)} warning(s).")
if not CONFIRM_INPUT:
    raise RuntimeError("STOPPED ON PURPOSE (not an error): look at the table, the alignments and the identity map above. "
                       "If these are the right sequences in the right clusters, tick CONFIRM_INPUT in Parameters and run again.")
print("Input confirmed.")

## 5. Setup and candidate pool
**What happens.**
- **`candidate pool first`:** all seed sequences search the whole database once (iterated, loose E-value `POOL_EVALUE`, up to `POOL_MAX_SEQS_PER_SEED` hits per seed). Every sequence found becomes the **pool**. All iterations then search only the pool, which is tiny compared with the database and takes seconds. Because E-values depend on database size, E-values from the pool are multiplied by (database residues / pool residues) so they mean what they would mean in the full database. After the last iteration, step 7 searches the full database once more with the final profiles and tells you what the pool missed.
- **`full database every iteration`:** no pool; each iteration searches the whole database. Most complete, and slow. `BUILD_INDEX` stores an MMseqs2 index next to the database once (extra disk, a long step), which makes each later search faster.

**Expect.** Pool: one loose, iterated database search, roughly as long as in notebook B (about 28 minutes per index pass for 5.7 M sequences at 8 GB; not measured for this loose setting; `POOL_ITERATIONS` 2 means two passes). Full mode with `BUILD_INDEX`: one long step now, then each search is faster (not measured). Output: pool size and the correction ratio.

**Parameters that matter.**
- `POOL_EVALUE` and `POOL_MAX_SEQS_PER_SEED`: how wide the net is. The pool must contain everything the later profiles could find; profiles are more sensitive than single seeds, so be generous. If the final check finds many sequences outside the pool, widen these or switch to full mode.
- `POOL_SENSITIVITY`, `POOL_ITERATIONS`: higher finds more remote sequences, takes longer.
- `SPLIT_MEMORY_LIMIT`: MMseqs2 splits the database to stay under this memory.

In [ ]:
#@title 5. Setup and candidate pool { display-mode: "form" }
POOL_MODE = SEARCH_SCOPE.startswith("candidate")
seed_params = {"db_id": INFO["id"], "seeds": SEEDS_KEY, "scope": SEARCH_SCOPE,
               "pool": [POOL_EVALUE, POOL_MAX_SEQS_PER_SEED, POOL_SENSITIVITY, POOL_ITERATIONS] if POOL_MODE else None}
POOL_DB = DB; RATIO = 1.0
meta = ck_load("setup", seed_params, expensive=True)
POOLDIR = WORK / "pool"
if POOL_MODE:
    if meta:
        keys = ck_table("setup", "keys").key.tolist()
    else:
        shutil.rmtree(POOLDIR, ignore_errors=True); POOLDIR.mkdir(parents=True)
        seedfa = POOLDIR / "seeds.fasta"
        seedfa.write_text("".join(f">{n}\n{s}\n" for n, s in zip(SEEDS.name, SEEDS.seq)))
        mm("createdb", seedfa, POOLDIR / "seedDB", "-v", "1", quiet=True)
        shutil.rmtree(TMP / "pool", ignore_errors=True)
        mm("search", POOLDIR / "seedDB", DB, POOLDIR / "res", TMP / "pool", "-s", POOL_SENSITIVITY, "-e", POOL_EVALUE,
           "--max-seqs", POOL_MAX_SEQS_PER_SEED, "--num-iterations", POOL_ITERATIONS, "--split-memory-limit", SPLIT_MEMORY_LIMIT,
           "--threads", THREADS, "-v", 3)
        ks = set()
        res, idx = POOLDIR / "res", POOLDIR / "res.index"
        data = res.read_bytes()
        for line in idx.read_text().splitlines():
            _, off, ln = line.split()[:3]
            for l in data[int(off):int(off) + int(ln)].decode(errors="ignore").split("\n"):
                if l.strip("\0"): ks.add(l.split("\t")[0].strip("\0"))
        keys = sorted(ks, key=int)
        if not keys: raise RuntimeError("The pool search found nothing. Loosen POOL_EVALUE or check the seeds and the database.")
    POOLDIR.mkdir(exist_ok=True)
    kf = POOLDIR / "keys.txt"
    kf.write_text("\n".join(keys) + "\n")
    for f in POOLDIR.glob("poolDB*"):
        f.unlink()
    mm("createsubdb", kf, DB, POOLDIR / "poolDB", "-v", 1, quiet=True)
    POOL_DB = POOLDIR / "poolDB"
    pool_res = db_residues(POOL_DB)
    RATIO = DB_RESIDUES / pool_res
    print(f"Pool: {len(keys):,} of {INFO['sequences']:,} sequences ({pool_res:,} residues); E-values from pool searches are multiplied by {RATIO:.1f}")
    if not meta:
        ck_save("setup", seed_params, tables={"keys": pd.DataFrame({"key": keys})}, extra={"ratio": RATIO})
else:
    if not meta:
        if BUILD_INDEX and not Path(str(DB) + ".idx").exists() and not Path(str(DB) + ".idx.index").exists():
            shutil.rmtree(TMP / "idx", ignore_errors=True)
            mm("createindex", DB, TMP / "idx", "--split-memory-limit", SPLIT_MEMORY_LIMIT, "--threads", THREADS, "-v", 3)
        ck_save("setup", seed_params, extra={"ratio": 1.0})
    print("Full-database mode: every iteration searches", INFO["name"])

## 6. Iterations
**What happens, per iteration k** (this cell defines the machinery; the next one runs it):
1. **Profile input.** For each cluster: the anchor, the extra seeds, and every member that is not contested and (if `PROFILE_MAX_EVALUE` > 0) has an E-value at least that good. If there are more than `MAX_PROFILE_SEQS`, near-identical sequences are thinned (the anchor and seeds always stay). Each member contributes only the **region that aligned to the previous profile**, so extra domains do not enter.
2. **Alignment and column filter.** MAFFT (or pairwise alignment to the anchor) aligns the region sequences with the anchor first. Columns are kept only if the anchor has a residue there and more than `MIN_OCCUPANCY` of the sequences do too. This is the "relevance region" you defined by choosing the anchor.
3. **Profile.** `mmseqs msa2profile` turns each filtered alignment into a profile (columns defined by the anchor).
4. **Search.** All profiles search the pool (or the database) with E-value `E_SCHEDULE[k]`, widened by `MARGIN` orders of magnitude so that runner-up clusters are visible. A hit must cover at least `MIN_PROFILE_COVERAGE` of the profile.
5. **Competition.** A hit becomes a member of its best cluster if the E-value there is `MARGIN` orders of magnitude (or more) better than in every other cluster. Otherwise it is **contested** between all clusters within `MARGIN` of the best. Every assignment is recomputed from scratch in each iteration, so a sequence can leave a cluster or become contested later.

**Expect.** Pool mode: seconds to a few minutes per iteration (MAFFT dominates for large clusters; not measured at real scale). Full mode: a full database search per iteration. Output per iteration: a table of cluster sizes, new / lost / switched / contested counts and E-value used, then the distance plot.

**Parameters that matter.**
- `E_SCHEDULE`, `MARGIN`: how fast clusters grow and how strict the conflict test is. A larger margin means more contested sequences, less risk of misassignment.
- `PROFILE_MAX_EVALUE`: keeps weak members out of profiles. The main protection against drift.
- `MIN_OCCUPANCY`: lower keeps more columns; higher keeps only the conserved core.
- `MAX_PROFILE_SEQS`, `MAFFT_MODE`: alignment time.
- `PROFILE_SENSITIVITY`, `MAX_HITS_PER_PROFILE`: search depth. If a profile hits `MAX_HITS_PER_PROFILE` the table warns that results were cut off.

In [ ]:
#@title 6a. Iteration machinery { display-mode: "form" }
MEMBER_COLS = ["cluster", "sid", "status", "evalue", "bits", "tstart", "tend", "tlen", "iter_added", "contested_with", "region", "seq"]
SEED_SEQS = set(SEEDS.seq)

PL = Align.PairwiseAligner()
PL.substitution_matrix = substitution_matrices.load("BLOSUM62")
PL.mode = "local"; PL.open_gap_score = -10; PL.extend_gap_score = -0.5

def clean_aa(s):
    return re.sub(r"[^A-Z]", "X", s.upper())

def local_to_anchor(anchor, seq):
    """Region of `seq` that aligns locally to `anchor`, and the gapped row over the anchor's columns."""
    a, s = clean_aa(anchor), clean_aa(seq)
    aln = PL.align(a, s)[0]
    row = ["-"] * len(a)
    lo, hi = len(s), 0
    for (a0, a1), (s0, s1) in zip(*aln.aligned):
        for k in range(a1 - a0): row[a0 + k] = s[s0 + k]
        lo, hi = min(lo, s0), max(hi, s1)
    return seq[lo:hi], "".join(row)

def initial_members():
    rows = []
    for _, r in SEEDS.iterrows():
        reg = r.seq if r.role == "anchor" else local_to_anchor(ANCHOR[r.cluster], r.seq)[0]
        rows.append({"cluster": r.cluster, "sid": r["name"], "status": r.role, "evalue": 0.0, "bits": 0.0, "tstart": 1, "tend": len(reg),
                     "tlen": len(r.seq), "iter_added": 0, "contested_with": "", "region": reg, "seq": r.seq})
    return pd.DataFrame(rows, columns=MEMBER_COLS)

def num_members(df):
    df = df.copy()
    for c in ("evalue", "bits"): df[c] = pd.to_numeric(df[c])
    for c in ("tstart", "tend", "tlen", "iter_added"): df[c] = pd.to_numeric(df[c]).astype(int)
    return df

def thin(rows, forced, cap, tag):
    """Reduce `rows` (DataFrame with sid, region, evalue) to about `cap` sequences; `forced` sids always stay."""
    if len(rows) <= cap:
        return rows
    fa = TMP / f"thin_{tag}.fa"
    fa.write_text("".join(f">{s}\n{r}\n" for s, r in zip(rows.sid, rows.region)))
    keep = None
    for ident in (0.95, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4):
        pref = TMP / f"thin_{tag}_c"
        shutil.rmtree(TMP / f"thintmp_{tag}", ignore_errors=True)
        mm("easy-cluster", fa, pref, TMP / f"thintmp_{tag}", "--min-seq-id", ident, "-c", 0.8, "--cov-mode", 1, "--threads", THREADS, "-v", 1, quiet=True)
        reps = {l[1:].split()[0] for l in open(f"{pref}_rep_seq.fasta") if l.startswith(">")} | set(forced)
        keep = reps
        if len(reps) <= cap: break
    out = rows[rows.sid.isin(keep)]
    if len(out) > cap:
        out = pd.concat([rows[rows.sid.isin(forced)], out[~out.sid.isin(forced)].sort_values("evalue").head(max(0, cap - len(forced)))])
    return out

MAFFT_ARGS = {"fast": ["--retree", "2", "--maxiterate", "0"], "auto": ["--auto"], "accurate": ["--localpair", "--maxiterate", "1000"]}

def msa_for_cluster(cl, rows, tag):
    """Aligned region sequences (anchor first), gapped, columns restricted to the anchor and to MIN_OCCUPANCY. Returns (names, rows, ncols)."""
    anchor = ANCHOR[cl]
    names, regs = [cl], [clean_aa(anchor)]
    for sid, reg in zip(rows.sid, rows.region):
        if sid == SEEDS[(SEEDS.cluster == cl) & (SEEDS.role == "anchor")].name.iloc[0]: continue
        names.append(sid); regs.append(clean_aa(reg))
    if len(names) == 1:
        return names, [regs[0]], len(regs[0])
    if USE_MAFFT:
        fa, out = TMP / f"msa_in_{tag}.fa", TMP / f"msa_out_{tag}.fa"
        fa.write_text("".join(f">{n}\n{s}\n" for n, s in zip(names, regs)))
        run([MAFFT, *MAFFT_ARGS[MAFFT_MODE], "--thread", THREADS, "--quiet", fa], stdout_path=out, quiet=True)
        from Bio import SeqIO
        recs = {r.id: str(r.seq).upper() for r in SeqIO.parse(out, "fasta")}
        grid = [recs[n] for n in names]
    else:
        grid = [regs[0]] + [local_to_anchor(regs[0], s)[1] for s in regs[1:]]
    arr = np.array([list(g) for g in grid])
    keep = arr[0] != "-"
    occ = (arr != "-").mean(axis=0)
    keep &= occ > MIN_OCCUPANCY
    if keep.sum() < 10:
        raise RuntimeError(f"Cluster {cl}: only {int(keep.sum())} alignment columns remain after the anchor and occupancy filters. "
                           "Lower MIN_OCCUPANCY or check the members.")
    return names, ["".join(r[keep]) for r in arr], int(keep.sum())

def write_msadb(path, msa_texts):
    off, idx = 0, []
    with open(path, "wb") as f:
        for k, t in enumerate(msa_texts):
            b = t.encode() + b"\0"
            f.write(b); idx.append(f"{k}\t{off}\t{len(b)}\n"); off += len(b)
    Path(str(path) + ".index").write_text("".join(idx))
    Path(str(path) + ".dbtype").write_bytes(struct.pack("<i", 11))

def build_profiles(M, tag):
    """Profile database for all clusters from the member table M. Returns (profile prefix, info table)."""
    texts, info = [], []
    for cl in CLUSTERS:
        sub = M[M.cluster == cl]
        ok = sub[sub.status.isin(["anchor", "seed"]) |
                 ((sub.status == "member") & ((sub.evalue <= PROFILE_MAX_EVALUE) if PROFILE_MAX_EVALUE > 0 else True))]
        forced = set(ok[ok.status.isin(["anchor", "seed"])].sid)
        ok2 = thin(ok, forced, MAX_PROFILE_SEQS, f"{tag}_{CNUM[cl]}")
        names, grid, ncol = msa_for_cluster(cl, ok2, f"{tag}_{CNUM[cl]}")
        texts.append("".join(f">{n}\n{g}\n" for n, g in zip(names, grid)))
        info.append({"cluster": cl, "profile_seqs": len(names), "profile_columns": ncol, "eligible": len(ok), "excluded_weak_or_contested": len(sub) - len(ok)})
    d = TMP / f"prof_{tag}"
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    write_msadb(d / "msa", texts)
    mm("msa2profile", d / "msa", d / "prof", "--match-mode", 0, "--threads", THREADS, "-v", 1, quiet=True)
    return d / "prof", pd.DataFrame(info)

HITFMT = "query,target,evalue,bits,qstart,qend,qlen,tstart,tend,tlen,tseq"

def profile_search(prof, target, e_search, tag):
    res, out = TMP / f"res_{tag}", TMP / f"hits_{tag}.tsv"
    for f in TMP.glob(f"res_{tag}*"): f.unlink()
    shutil.rmtree(TMP / f"sr_{tag}", ignore_errors=True)
    mm("search", prof, target, res, TMP / f"sr_{tag}", "-e", e_search, "-s", PROFILE_SENSITIVITY, "--max-seqs", MAX_HITS_PER_PROFILE,
       "--split-memory-limit", SPLIT_MEMORY_LIMIT, "--threads", THREADS, "-v", 1, quiet=True)
    mm("convertalis", prof, target, res, out, "--format-output", HITFMT, "--threads", THREADS, "-v", 1, quiet=True)
    cols = HITFMT.split(",")
    if not out.exists() or out.stat().st_size == 0:
        return pd.DataFrame(columns=cols)
    return pd.read_csv(out, sep="\t", names=cols, dtype={"query": str, "target": str, "tseq": str})

def compete(hits, e_k, ratio, prev):
    """Assign hits to clusters. Returns (member table, n_truncated_profiles)."""
    h = hits.copy()
    h["e"] = np.maximum(h.evalue.astype(float) * ratio, 1e-300)
    h["pcov"] = (h.qend - h.qstart + 1) / h.qlen
    h = h[(h.pcov >= MIN_PROFILE_COVERAGE) & (~h.tseq.isin(SEED_SEQS))]
    h = h.sort_values("e").drop_duplicates(["query", "target"])
    trunc = int((hits.groupby("query").size() >= MAX_HITS_PER_PROFILE).sum()) if len(hits) else 0
    if h.empty:
        return pd.DataFrame(columns=MEMBER_COLS), trunc
    h["le"] = np.log10(h.e)
    piv = h.pivot(index="target", columns="query", values="le").reindex(columns=CLUSTERS)
    L = piv.to_numpy(dtype=float).copy(); L[np.isnan(L)] = np.inf
    best = L.min(axis=1)
    keep = best <= math.log10(e_k)                       # at least one cluster passes the iteration's E-value
    within = L <= (best[:, None] + MARGIN)
    rows = []
    prev_added = {(r.cluster, r.sid): r.iter_added for r in prev.itertuples()} if len(prev) else {}
    info = h.set_index(["query", "target"])
    for t, bst, w, ok in zip(piv.index, L.argmin(axis=1), within, keep):
        if not ok: continue
        cls = [CLUSTERS[i] for i in np.where(w)[0]]
        status = "member" if len(cls) == 1 else "contested"
        for cl in cls:
            r = info.loc[(cl, t)]
            others = [CNUM[c] for c in cls if c != cl]
            sep = "" if max(CNUM.values()) < 10 else "_"
            rows.append({"cluster": cl, "sid": t, "status": status, "evalue": float(r.e), "bits": float(r.bits), "tstart": int(r.tstart),
                         "tend": int(r.tend), "tlen": int(r.tlen), "iter_added": None,
                         "contested_with": sep.join(map(str, others)), "region": r.tseq[int(r.tstart) - 1:int(r.tend)], "seq": r.tseq})
    return pd.DataFrame(rows, columns=MEMBER_COLS), trunc

def next_members(prev, e_k, ratio, k, target, tag):
    """One iteration: profiles from `prev`, search `target`, competition. Returns (new member table, stats table, profile info)."""
    t0 = time.time()
    prof, pinfo = build_profiles(prev, tag)
    t1 = time.time()
    hits = profile_search(prof, target, e_k * 10 ** MARGIN * ratio, tag)
    t2 = time.time()
    new, trunc = compete(hits, e_k, ratio, prev)
    fixed = prev[prev.status.isin(["anchor", "seed"])].copy()
    pm = {(r.cluster, r.sid): r.iter_added for r in prev.itertuples() if r.status in ("member", "contested")}
    new["iter_added"] = [pm.get((c, s), k) for c, s in zip(new.cluster, new.sid)]
    M = pd.concat([fixed, new], ignore_index=True)[MEMBER_COLS]
    prev_mem = prev[prev.status == "member"]
    pm_cl = dict(zip(prev_mem.sid, prev_mem.cluster))
    stats = []
    for cl in CLUSTERS:
        cur = M[(M.cluster == cl) & (M.status == "member")]
        was = prev_mem[prev_mem.cluster == cl]
        cur_all = M[M.sid.isin(cur.sid)]
        stats.append({"iteration": k, "cluster": cl, "E_value": e_k, "members": len(cur), "seeds": int(M[(M.cluster == cl) & M.status.isin(["anchor", "seed"])].shape[0]),
                      "new": int((~cur.sid.isin(was.sid) & ~cur.sid.isin([s for s, c in pm_cl.items() if c != cl])).sum()),
                      "switched_in": int(cur.sid.isin([s for s, c in pm_cl.items() if c != cl]).sum()),
                      "lost": int((~was.sid.isin(cur.sid)).sum()),
                      "contested": int(((M.cluster == cl) & (M.status == "contested")).sum()),
                      "search_hits": int((hits["query"] == cl).sum()),
                      "seconds_profile": round(t1 - t0), "seconds_search": round(t2 - t1)})
    st = pd.DataFrame(stats).merge(pinfo, on="cluster")
    st.attrs["truncated"] = trunc
    return M, st, pinfo

print("Iteration machinery defined.")

## 6b. Run the iterations
**What happens.** Starts from the seeds (iteration 0) and runs iterations 1 to `ITERATIONS`. An iteration whose checkpoint matches the current settings is loaded instead of recomputed; iterations beyond the last saved one are run. Each iteration prints its table. **Contested** counts are the number of sequences held out of the profiles because two or more clusters match them about equally well. After each iteration the distance plot appears (see 6c).

**Expect.** Pool mode: seconds to minutes per iteration (not measured at real scale). Full mode: each iteration is a database search. In full mode a changed setting stops the run rather than silently redoing an expensive search; tick `OVERRIDE_CHECKPOINT` or `FORCE_RERUN` as the message says.

**Parameters that matter.** `ITERATIONS` (raise it and re-run to continue), `E_SCHEDULE`, `MARGIN`, `PROFILE_MAX_EVALUE`.

In [ ]:
#@title 6b. Run the iterations { display-mode: "form" }
HISTORY = {0: initial_members()}
STATS = []
E_OF = lambda k: ES[min(k, len(ES)) - 1]
prev_step = "setup"
for k in range(1, ITERATIONS + 1):
    step = f"iter{k:02d}"
    e_k = E_OF(k)
    params = {"upstream": KEYS[prev_step], "k": k, "E": e_k, "margin": MARGIN, "profile_max_evalue": PROFILE_MAX_EVALUE,
              "min_occupancy": MIN_OCCUPANCY, "min_cov": MIN_PROFILE_COVERAGE, "msa": ("mafft:" + MAFFT_MODE) if USE_MAFFT else "pairwise",
              "profile_sens": PROFILE_SENSITIVITY, "max_hits": MAX_HITS_PER_PROFILE, "max_profile_seqs": MAX_PROFILE_SEQS, "ratio": round(RATIO, 4)}
    meta = ck_load(step, params, expensive=not POOL_MODE)
    if meta:
        M = num_members(ck_table(step, "members")); st = ck_table(step, "stats")
        for c in st.columns:
            if c not in ("cluster",): st[c] = pd.to_numeric(st[c])
        trunc = int(meta["extra"].get("truncated", 0))
    else:
        print(f"\n=== Iteration {k}: E-value {e_k:g}, margin {MARGIN:g} orders of magnitude ===")
        M, st, _ = next_members(HISTORY[k - 1], e_k, RATIO, k, POOL_DB, step)
        trunc = st.attrs.get("truncated", 0)
        ck_save(step, params, tables={"members": M, "stats": st}, extra={"truncated": trunc})
    HISTORY[k] = M; STATS.append(st); prev_step = step
    nc = len(M[M.status == "contested"].sid.unique())
    print(f"\nIteration {k} (E {e_k:g}): {int((M.status == 'member').sum())} members in total, {nc} CONTESTED sequences held out of the profiles")
    display(st.drop(columns=["E_value", "iteration"]))
    if trunc: print(f"WARNING: {trunc} profile(s) reached MAX_HITS_PER_PROFILE; the hit list is cut off. Raise it.")
STATS = pd.concat(STATS, ignore_index=True)

## 6c. Distance plots
**What happens.** For each iteration a figure shows how far apart the clusters are, using a sample of at most `DIST_SAMPLE` sequences per cluster (anchor and seeds always included, newest members preferred) cut to the region that aligned to the profile. All pairs in the sample are aligned with MMseqs2 (exhaustive, with the substitution matrix `DIST_MATRIX`). The distance is `1 - identity` over the shorter region (`DIST_METRIC = identity`), or `1 - score / lower self-score` (`normalised score`). Pairs MMseqs2 cannot align count as distance 1.
- **Top row, one panel per cluster:** histogram of distances *within* the cluster (cluster colour) against distances from its members to *all other clusters* (grey). Red lines are the same for pairs that involve at least one **new** member of this iteration. Well separated clusters show the grey histogram far to the right of the coloured one.
- **Bottom left:** every sampled member as a point: mean distance to the rest of its own cluster (x) against the mean distance to the nearest other cluster (y). Points **below the diagonal** are closer to another cluster than to their own. New members have a red edge; contested sequences are black crosses.
- **Bottom right:** a 2-D map of the sample (classical multidimensional scaling of the same distances). It is a distortion of the real geometry; use it for orientation only.

**Expect.** Seconds for a few hundred sequences (not measured). Plots are saved to `OUT/plots/`. By default only the latest iteration is shown; `SHOW_ALL_ITERATION_PLOTS` shows all.

**Parameters that matter.** `DIST_SAMPLE` (cost grows with its square times the number of clusters), `DIST_METRIC`, `DIST_MATRIX` (PAM30 emphasises close relatives, BLOSUM45 and VTML80 remote ones; identity ignores the matrix except for the alignment itself).

In [ ]:
#@title 6c. Distance plots { display-mode: "form" }
PLOTDIR = OUT / "plots"; PLOTDIR.mkdir(exist_ok=True)
PALETTE = plt.get_cmap("tab10").colors

def sample_for_plot(M, k, cap):
    rng = np.random.default_rng(1)
    parts = []
    for cl in CLUSTERS:
        mem = M[(M.cluster == cl) & M.status.isin(["anchor", "seed", "member"])].drop_duplicates("sid")
        fixed = mem[mem.status.isin(["anchor", "seed"])]
        rest = mem[mem.status == "member"]
        new = rest[rest.iter_added == k]; old = rest[rest.iter_added != k]
        room = max(0, cap - len(fixed))
        n_new = min(len(new), max(room // 2, room - len(old)))
        pick = pd.concat([new.sample(n_new, random_state=1) if n_new else new.head(0),
                          old.sample(min(len(old), room - n_new), random_state=1) if room - n_new > 0 else old.head(0)])
        df = pd.concat([fixed, pick]); df = df.assign(group=cl, new=df.iter_added == k)
        df.loc[df.status.isin(["anchor", "seed"]), "new"] = False
        parts.append(df)
    con = M[M.status == "contested"].drop_duplicates("sid")
    con = con.sample(min(len(con), max(cap // 2, 10)), random_state=1) if len(con) else con
    parts.append(con.assign(group="contested", new=False))
    S = pd.concat(parts, ignore_index=True)
    S = S.drop_duplicates("sid").reset_index(drop=True)
    return S

def distance_matrix(S, tag):
    fa = TMP / f"dist_{tag}.fa"
    fa.write_text("".join(f">u{i}\n{r}\n" for i, r in enumerate(S.region)))
    out = TMP / f"dist_{tag}.tsv"
    shutil.rmtree(TMP / f"dtmp_{tag}", ignore_errors=True)
    mm("easy-search", fa, fa, out, TMP / f"dtmp_{tag}", "--exhaustive-search", 1, "-e", 1e6, "--max-seqs", 100000, "--alignment-mode", 3,
       "--sub-mat", f"aa:{DIST_MATRIX},nucl:nucleotide.out", "--min-seq-id", 0, "-c", 0, "--threads", THREADS, "-v", 1,
       "--format-output", "query,target,fident,alnlen,bits", quiet=True)
    n = len(S)
    ln = S.region.str.len().to_numpy()
    D = np.ones((n, n))
    if out.exists() and out.stat().st_size:
        t = pd.read_csv(out, sep="\t", names=["q", "t", "fident", "alnlen", "bits"])
        qi = t.q.str[1:].astype(int).to_numpy(); ti = t.t.str[1:].astype(int).to_numpy()
        if DIST_METRIC == "identity":
            val = 1 - (t.fident * t.alnlen).to_numpy() / np.minimum(ln[qi], ln[ti])
        else:
            selfb = np.zeros(n)
            m = qi == ti
            selfb[qi[m]] = t.bits.to_numpy()[m]
            val = 1 - t.bits.to_numpy() / np.maximum(np.minimum(selfb[qi], selfb[ti]), 1)
        val = np.clip(val, 0, 1)
        Dn = np.ones((n, n))
        np.minimum.at(Dn, (qi, ti), val)
        D = np.minimum(Dn, Dn.T)
    np.fill_diagonal(D, 0)
    return D

def plot_iteration(k, M, show=True):
    S = sample_for_plot(M, k, DIST_SAMPLE)
    D = distance_matrix(S, f"it{k}")
    nC = len(CLUSTERS); ncols = max(nC, 2)
    fig = plt.figure(figsize=(3.6 * ncols, 7.6))
    gs = fig.add_gridspec(2, 2 * ncols)
    col = {c: PALETTE[i % 10] for i, c in enumerate(CLUSTERS)}
    grp = S.group.to_numpy(); isnew = S.new.to_numpy(bool)
    bins = np.linspace(0, 1, 31)
    own_mean = np.full(len(S), np.nan); other_min = np.full(len(S), np.nan)
    for i, cl in enumerate(CLUSTERS):
        ax = fig.add_subplot(gs[0, 2 * i:2 * i + 2])
        ic = np.where(grp == cl)[0]; io = np.where((grp != cl) & (grp != "contested"))[0]
        w = D[np.ix_(ic, ic)][np.triu_indices(len(ic), 1)]
        c = D[np.ix_(ic, io)].ravel() if len(io) else np.array([])
        nw = (isnew[ic][:, None] | isnew[ic][None, :])[np.triu_indices(len(ic), 1)]
        nc_ = (isnew[ic][:, None] | isnew[io][None, :]).ravel() if len(io) else np.array([], bool)
        for data, kw in ((w, dict(color=col[cl], lw=1.8, label="within")), (c, dict(color="0.45", lw=1.8, label="to other clusters"))):
            if len(data): ax.hist(data, bins=bins, density=True, histtype="step", **kw)
        if nw.any(): ax.hist(w[nw], bins=bins, density=True, histtype="step", color="red", lw=1.2, label="within, new involved")
        if len(c) and nc_.any(): ax.hist(c[nc_], bins=bins, density=True, histtype="step", color="red", lw=1.2, ls="--", label="other, new involved")
        ax.set_title(f"{CNUM[cl]}: {cl} (n={len(ic)})", fontsize=9); ax.set_xlabel("distance"); ax.set_xlim(0, 1)
        if i == 0: ax.legend(fontsize=6)
        for j in ic:
            own = [x for x in ic if x != j]
            own_mean[j] = D[j, own].mean() if own else np.nan
            others = [D[j, np.where(grp == o)[0]].mean() for o in CLUSTERS if o != cl and (grp == o).any()]
            other_min[j] = min(others) if others else np.nan
    ax = fig.add_subplot(gs[1, :ncols])
    lim = [0, 1]; ax.plot(lim, lim, color="0.6", lw=0.8)
    for cl in CLUSTERS:
        m = (grp == cl) & ~isnew; mn = (grp == cl) & isnew
        ax.scatter(own_mean[m], other_min[m], s=14, color=col[cl], alpha=0.7, label=cl)
        ax.scatter(own_mean[mn], other_min[mn], s=28, color=col[cl], edgecolor="red", linewidth=1.2)
    if (grp == "contested").any():
        cm = np.where(grp == "contested")[0]
        cm_own = [np.nan] * len(cm)
        for n_, j in enumerate(cm):
            ms = [D[j, np.where(grp == o)[0]].mean() for o in CLUSTERS if (grp == o).any()]
            ax.scatter([np.sort(ms)[0]], [np.sort(ms)[1] if len(ms) > 1 else np.nan], marker="x", color="black", s=22, label="contested" if n_ == 0 else None)
    ax.set_xlabel("mean distance to own cluster (contested: to nearest cluster)"); ax.set_ylabel("mean distance to nearest other cluster")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_title("below the diagonal = closer to another cluster than to its own", fontsize=8); ax.legend(fontsize=6)
    ax = fig.add_subplot(gs[1, ncols:])
    n = len(S)
    if n >= 3:
        J = np.eye(n) - 1.0 / n; B = -0.5 * J @ (D ** 2) @ J
        ev, evec = np.linalg.eigh(B); order = np.argsort(ev)[::-1][:2]
        X = evec[:, order] * np.sqrt(np.maximum(ev[order], 0))
        for cl in CLUSTERS:
            m = (grp == cl) & ~isnew; mn = (grp == cl) & isnew
            ax.scatter(X[m, 0], X[m, 1], s=14, color=col[cl], alpha=0.7)
            ax.scatter(X[mn, 0], X[mn, 1], s=28, color=col[cl], edgecolor="red", linewidth=1.2)
            a = S.index[(S.group == cl) & (S.status == "anchor")]
            ax.scatter(X[a, 0], X[a, 1], marker="*", s=140, color=col[cl], edgecolor="black", zorder=5)
        cm = grp == "contested"
        ax.scatter(X[cm, 0], X[cm, 1], marker="x", color="black", s=22)
    ax.set_title("2-D map (MDS of the same distances); stars = anchors, red edge = new", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Iteration {k}: E-value {E_OF(k):g}, metric {DIST_METRIC}, matrix {DIST_MATRIX}", fontsize=10)
    plt.tight_layout()
    path = PLOTDIR / f"iteration_{k:02d}.png"
    fig.savefig(path, dpi=130)
    if show: plt.show()
    plt.close(fig)
    return path

for k in range(1, ITERATIONS + 1):
    show = SHOW_ALL_ITERATION_PLOTS or k == ITERATIONS
    p = plot_iteration(k, HISTORY[k], show=show)
print("Plots saved in", PLOTDIR)

## 7. Final check against the full database (pool mode)
**What happens.** Runs the equivalent of one more iteration, building profiles from the last iteration's members, but searching the **whole database** instead of the pool. It reports how many members and contested sequences the pool-based result is missing (or has extra). Differences have two causes: sequences outside the pool, and the extra growth of one more iteration. If the numbers are large relative to the cluster sizes, the pool was too narrow: widen `POOL_EVALUE` and `POOL_MAX_SEQS_PER_SEED`, or use full-database mode. These sequences are written to separate FASTA files (`*_outside_pool.fasta`); they are not added to the main cluster files.

**Expect.** One full database search with profiles: like a notebook B search or longer (not measured). Skipped in full-database mode or if `FINAL_CHECK` is off.

**Parameters that matter.** `FINAL_CHECK`, and the pool settings in step 5.

In [ ]:
#@title 7. Final check { display-mode: "form" }
LAST = HISTORY[ITERATIONS]
OUTSIDE = None
if POOL_MODE and FINAL_CHECK:
    kf = ITERATIONS + 1
    params = {"upstream": KEYS[f"iter{ITERATIONS:02d}"], "E": E_OF(kf), "margin": MARGIN, "db_id": INFO["id"], "profile_max_evalue": PROFILE_MAX_EVALUE,
              "min_occupancy": MIN_OCCUPANCY, "min_cov": MIN_PROFILE_COVERAGE, "msa": ("mafft:" + MAFFT_MODE) if USE_MAFFT else "pairwise",
              "profile_sens": PROFILE_SENSITIVITY, "max_hits": MAX_HITS_PER_PROFILE, "max_profile_seqs": MAX_PROFILE_SEQS}
    meta = ck_load("final_check", params, expensive=True)
    if meta:
        MF = num_members(ck_table("final_check", "members"))
    else:
        MF, stf, _ = next_members(LAST, E_OF(kf), 1.0, kf, DB, "final")
        ck_save("final_check", params, tables={"members": MF, "stats": stf})
    a = set(zip(LAST.cluster, LAST.sid, LAST.status)); b = set(zip(MF.cluster, MF.sid, MF.status))
    only_full = MF[[x in (b - a) for x in zip(MF.cluster, MF.sid, MF.status)]]
    only_pool = [x for x in a - b if x[2] in ("member", "contested")]
    OUTSIDE = only_full[only_full.status.isin(["member", "contested"])]
    print(f"Full-database check (one more iteration at E {E_OF(kf):g}): {len(OUTSIDE)} (cluster, sequence) assignments not in the pool-based result, {len(only_pool)} in the pool-based result that the full database check does not reproduce.")
    summ = pd.DataFrame({"cluster": CLUSTERS,
                         "pool result (members)": [int(((LAST.cluster == c) & (LAST.status == "member")).sum()) for c in CLUSTERS],
                         "full check (members)": [int(((MF.cluster == c) & (MF.status == "member")).sum()) for c in CLUSTERS],
                         "extra members": [int(((OUTSIDE.cluster == c) & (OUTSIDE.status == "member")).sum()) for c in CLUSTERS],
                         "extra contested": [int(((OUTSIDE.cluster == c) & (OUTSIDE.status == "contested")).sum()) for c in CLUSTERS]})
    display(summ)
else:
    print("Final check skipped.")

## 8. Output
**What happens.** Writes, under `OUT_DIR` (default `work_C/results`):
- `clusters/<label>.fasta`: one file per cluster from the last iteration. Anchor first (`anchor`), then extra seeds, then members (E-value and the iteration in which they joined). Sequences contested between clusters appear in the file of every cluster involved and are named `contested_a<n>_<original name>`, where `<n>` is the number of the other cluster(s) (`a2`: contested with cluster 2; `a23`: with 2 and 3).
- `members.tsv`, `contested.tsv`, `iteration_stats.tsv`, `plots/iteration_XX.png`.
- `*_outside_pool.fasta` for the final check, if it ran.
- `cluster_growth_results.zip` with all of this (offered for download in Colab).

**Expect.** Seconds. Output: counts per cluster, including distinct taxonomy ids if the database has a mapping (UniRef50 entries are representatives, so these are not species counts).

**Parameters that matter.** `FASTA_REGION_ONLY` writes only the aligned region instead of the full sequence.

In [ ]:
#@title 8. Output { display-mode: "form" }
def taxids_of(names):
    """Map sequence names to taxids with the database's lookup and mapping files (empty dict if the database has no mapping)."""
    mp = Path(str(DB) + "_mapping")
    if not mp.exists(): return {}
    need = set(names); need |= {n.split("|")[1] for n in names if n.count("|") >= 2}
    keys = {}
    for ch in pd.read_csv(str(DB) + ".lookup", sep="\t", names=["key", "name", "f"], usecols=[0, 1], chunksize=2_000_000, dtype=str):
        for key, nm in ch[ch.name.isin(need)][["key", "name"]].itertuples(index=False): keys[key] = nm
    tx = {}
    for ch in pd.read_csv(mp, sep="\t", names=["key", "taxid"], chunksize=2_000_000, dtype=str):
        for key, t in ch[ch.key.isin(keys)].itertuples(index=False): tx[keys[key]] = t
    out = {}
    for n in names:
        out[n] = tx.get(n) or (tx.get(n.split("|")[1]) if n.count("|") >= 2 else None)
    return out

def tag_for(row):
    return f"contested_a{row.contested_with}_{row.sid}"

def write_clusters(Mx, folder, only_new_vs=None, suffix=""):
    folder.mkdir(parents=True, exist_ok=True)
    for cl in CLUSTERS:
        with open(folder / f"{cl}{suffix}.fasta", "w", newline="\n") as f:
            sub = Mx[Mx.cluster == cl]
            order = {"anchor": 0, "seed": 1, "member": 2, "contested": 3}
            sub = sub.assign(o=sub.status.map(order)).sort_values(["o", "evalue", "sid"])
            for r in sub.itertuples():
                seq = r.region if FASTA_REGION_ONLY else r.seq
                if r.status == "contested":
                    h = f"{tag_for(r)} E={r.evalue:.2g}"
                elif r.status in ("anchor", "seed"):
                    h = f"{r.sid} {r.status}"
                else:
                    h = f"{r.sid} E={r.evalue:.2g} added_in_iteration={r.iter_added}"
                f.write(f">{h}\n{seq}\n")

write_clusters(LAST, OUT / "clusters")
allm = LAST.drop(columns=["seq"]).copy()
TX = taxids_of(list(LAST[LAST.status.isin(["member", "contested"])].sid.unique()))
allm["taxid"] = allm.sid.map(TX) if TX else ""
allm.to_csv(OUT / "members.tsv", sep="\t", index=False)
cont = LAST[LAST.status == "contested"].groupby("sid").agg(clusters=("cluster", lambda x: ",".join(f"{CNUM[c]}" for c in x)),
                                                             best_evalue=("evalue", "min")).reset_index()
cont.to_csv(OUT / "contested.tsv", sep="\t", index=False)
STATS.to_csv(OUT / "iteration_stats.tsv", sep="\t", index=False)
if OUTSIDE is not None and len(OUTSIDE):
    write_clusters(OUTSIDE, OUT / "clusters_outside_pool", suffix="_outside_pool")
zipf = shutil.make_archive(str(OUT.parent / "cluster_growth_results"), "zip", str(OUT))

print(f"Results in {OUT}")
final = []
for cl in CLUSTERS:
    mem = LAST[(LAST.cluster == cl) & (LAST.status == "member")]
    final.append({"cluster": f"{CNUM[cl]}: {cl}", "anchor+seeds": int(LAST[(LAST.cluster == cl) & LAST.status.isin(['anchor', 'seed'])].shape[0]),
                  "members": len(mem), "contested": int(((LAST.cluster == cl) & (LAST.status == 'contested')).sum()),
                  "distinct taxids": len({TX.get(s) for s in mem.sid} - {None}) if TX else "n/a"})
display(pd.DataFrame(final))
print("\nPer iteration (all clusters):")
display(STATS.groupby("iteration").agg(E_value=("E_value", "first"), members=("members", "sum"), new=("new", "sum"), switched_in=("switched_in", "sum"),
                                         lost=("lost", "sum")).join(
        pd.Series({k: len(HISTORY[k][HISTORY[k].status == "contested"].sid.unique()) for k in range(1, ITERATIONS + 1)}, name="contested sequences")))
print("Zip:", zipf)
if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zipf)